In [1]:
# Libraries
import numpy as np
import pandas as pd
from scipy import stats

In [3]:
# Master tables.
masterTableHuman = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/humanParalogy.txt", sep="\t")
masterTableMouse = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/mouseParalogy.txt", sep="\t")

# Add Expression Profile Information

In [4]:
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")

# Merge the paralogue table with the expression profile data. Have to do this twice for the parental and duaghter copies.
masterTableHuman = masterTableHuman.merge(gtexEP.add_prefix("Parental "), on="Gene stable ID", how="left")
masterTableHuman = masterTableHuman.merge(gtexEP.add_prefix("Daughter "), left_on="Human paralogue gene stable ID", right_on="Gene stable ID", how="left")

# Merge the paralogue table with the expression profile data. Have to do this twice for the parental and duaghter copies.
masterTableMouse = masterTableMouse.merge(emtabEP.add_prefix("Parental "), on="Gene stable ID", how="left")
masterTableMouse = masterTableMouse.merge(emtabEP.add_prefix("Daughter "), left_on="Mouse paralogue gene stable ID", right_on="Gene stable ID", how="left")

# Calculate Distances

In [5]:
def calcTEC(geneOne, geneTwo):
    # Turns the expression profiles into a binary vector that tells us whether a gene is expressed (TPM > 1) or not in a tissue. Turn the dataframe into a series.
    humanOrthoBinary = (geneOne[:-1] > 1)
    mouseOrthoBinary = (geneTwo[:-1] > 1)

    # Finds the number of tissues that are found in one species and not in the other.
    humanOnlyTissueNum = (humanOrthoBinary & ~mouseOrthoBinary).sum()
    mouseOnlyTissueNum = (mouseOrthoBinary & ~humanOrthoBinary).sum()

    # Using the binary vector, we can calculate the total number of tissues the gene is expressed in.
    humanTotalTissue = humanOrthoBinary.sum()
    mouseTotalTissue = mouseOrthoBinary.sum()

    # If any gene is not expressed in any tissue, the TEC formula will output an error. We handle this case by outputting NaN.
    if humanTotalTissue == 0 or mouseTotalTissue == 0:
        return np.nan
    else:
        return ((humanOnlyTissueNum / humanTotalTissue) + (mouseOnlyTissueNum / mouseTotalTissue)) / 2

In [6]:
# Grabs the parental and daughter copy expression profiles.
parentalEPHuman = masterTableHuman.filter(like="Parental")
daughterEPHuman = masterTableHuman.filter(like="Daughter")

# Calculates Euclidean distance.
masterTableHuman["EuclidDist"] = np.linalg.norm(parentalEPHuman.select_dtypes(include="number").to_numpy() - daughterEPHuman.select_dtypes(include="number").to_numpy(), axis=1)

# Calculates Euclidean distance with Euclidean normalization applied. First calculates the Euclidean norm of each row (axis=1), then divides each "index" (row) by its respective Euclidean norm. 
masterTableHuman["EuclidDistNorm"] = np.linalg.norm(parentalEPHuman.select_dtypes(include="number").div(np.linalg.norm(parentalEPHuman.select_dtypes(include="number"), axis=1), axis=0).to_numpy() - daughterEPHuman.select_dtypes(include="number").div(np.linalg.norm(daughterEPHuman.select_dtypes(include="number"), axis=1), axis=0).to_numpy(), axis=1)

# Calculates Euclidean distance with log2 transformation applied.
masterTableHuman["EuclidDistLog"] = np.linalg.norm(np.log2(parentalEPHuman.select_dtypes(include="number") + 1).to_numpy() - np.log2(daughterEPHuman.select_dtypes(include="number") + 1).to_numpy(), axis=1)                                                                                                                                                                                                                                                                                                                                                                                                                                                                

# Calculates Pearson distance. The "corrwith" function requires dataframes to have the same column name. That's why I had to rename daughterEPHuman's columns to parentalEPHuman'
masterTableHuman["PearDist"] = 1 - parentalEPHuman.select_dtypes(include="number").corrwith(daughterEPHuman.select_dtypes(include="number").rename(columns=dict(zip(daughterEPHuman.columns, parentalEPHuman.columns))), axis=1, method="pearson").to_numpy()

# Iterates through each row in both dataframes and calculates their TEC score.
masterTableHuman["TEC"] = [calcTEC(parentCopy, daughterCopy) for parentCopy, daughterCopy in zip(parentalEPHuman.select_dtypes(include="number").to_numpy(), daughterEPHuman.select_dtypes(include="number").to_numpy())]

In [7]:
# Grabs the parental and daughter copy expression profiles.
parentalEPMouse = masterTableMouse.filter(like="Parental")
daughterEPMouse = masterTableMouse.filter(like="Daughter")

# Calculates Euclidean distance.
masterTableMouse["EuclidDist"] = np.linalg.norm(parentalEPMouse.select_dtypes(include="number").to_numpy() - daughterEPMouse.select_dtypes(include="number").to_numpy(), axis=1)

# Calculates Euclidean distance with Euclidean normalization applied. First calculates the Euclidean norm of each row (axis=1), then divides each "index" (row) by its respective Euclidean norm. 
masterTableMouse["EuclidDistNorm"] = np.linalg.norm(parentalEPMouse.select_dtypes(include="number").div(np.linalg.norm(parentalEPMouse.select_dtypes(include="number"), axis=1), axis=0).to_numpy() - daughterEPMouse.select_dtypes(include="number").div(np.linalg.norm(daughterEPMouse.select_dtypes(include="number"), axis=1), axis=0).to_numpy(), axis=1)

# Calculates Euclidean distance with log2 transformation applied.
masterTableMouse["EuclidDistLog"] = np.linalg.norm(np.log2(parentalEPMouse.select_dtypes(include="number") + 1).to_numpy() - np.log2(daughterEPMouse.select_dtypes(include="number") + 1).to_numpy(), axis=1)                                                                                                                                                                                                                                                                                                                                                                                                                                                                

# Calculates Pearson distance. The "corrwith" function requires dataframes to have the same column name. That's why I had to rename daughterEPMouse's columns to parentalEPMouse'
masterTableMouse["PearDist"] = 1 - parentalEPMouse.select_dtypes(include="number").corrwith(daughterEPMouse.select_dtypes(include="number").rename(columns=dict(zip(daughterEPMouse.columns, parentalEPMouse.columns))), axis=1, method="pearson").to_numpy()

# Iterates through each row in both dataframes and calculates their TEC score.
masterTableMouse["TEC"] = [calcTEC(parentCopy, daughterCopy) for parentCopy, daughterCopy in zip(parentalEPMouse.select_dtypes(include="number").to_numpy(), daughterEPMouse.select_dtypes(include="number").to_numpy())]

# Number of Duplicates

In [8]:
humanPairs = masterTableHuman.iloc[:, 0:2]
mousePairs = masterTableMouse.iloc[:, 0:2]

In [9]:
masterTableHuman = masterTableHuman.merge(humanPairs.groupby("Gene stable ID").count(), left_on="Gene stable ID", right_index=True, how="left").rename(columns={"Human paralogue gene stable ID_y": "Number of Duplicates"})
masterTableMouse = masterTableMouse.merge(mousePairs.groupby("Gene stable ID").count(), left_on="Gene stable ID", right_index=True, how="left").rename(columns={"Mouse paralogue gene stable ID_y": "Number of Duplicates"})

# Exons

In [10]:
humanExons = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/humanExons.txt", sep="\t")
mouseExons = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/mouseExons.txt", sep="\t")

humanExonsGrouped = humanExons.groupby("Gene stable ID")["Exon stable ID"].agg(", ".join)
mouseExonsGrouped = mouseExons.groupby("Gene stable ID")["Exon stable ID"].agg(", ".join)

In [11]:
masterTableHuman = masterTableHuman.merge(humanExonsGrouped, on="Gene stable ID", how="left")
masterTableMouse = masterTableMouse.merge(mouseExonsGrouped, on="Gene stable ID", how="left")

# Identifying In/Out-Paralogs

In [12]:
masterTableMouse["Paralogue last common ancestor with Mouse"].unique()

<ArrowStringArray>
[                                      nan,
                               'Bilateria',
 'Mus musculus reference (CL57BL6) strain',
                                     'Mus',
                                 'Murinae',
                                'Muroidea',
                                'Rodentia',
                                'Eutheria',
                           'Gnathostomata',
                              'Vertebrata',
                                'Mammalia',
                           'Boreoeutheria',
                            'Euteleostomi',
                                'Chordata',
                                 'Amniota',
                            'Opisthokonta',
                                  'Theria',
                           'Sarcopterygii',
                               'Tetrapoda',
                               'Myomorpha',
                        'Euarchontoglires',
                                  'Glires']
Length: 22, d

In [13]:
humanSpecific = ["Primates", "Haplorrhini", "Simiiformes", "Catarrhini", "Hominidae", "Homininae", "Homo sapiens"]
mouseSpecific = ["Glires", "Rodentia", "Myomorpha", "Muroidea", "Murinae", "Mus", "Mus musculus reference (CL57BL6) strain"]

humanParalogyConditions = [
    masterTableHuman["Paralogue last common ancestor with Human"].isin(humanSpecific),
    (~masterTableHuman["Paralogue last common ancestor with Human"].isin(humanSpecific)) & (~masterTableHuman["Paralogue last common ancestor with Human"].isna()),
]
mouseParalogyConditions = [
    masterTableMouse["Paralogue last common ancestor with Mouse"].isin(mouseSpecific),
    (~masterTableMouse["Paralogue last common ancestor with Mouse"].isin(mouseSpecific)) & (~masterTableMouse["Paralogue last common ancestor with Mouse"].isna()),
]

paralogyChoices = [
    "in-paralog",
    "out-paralog"
]

masterTableHuman["Paralogy Type"] = np.select(humanParalogyConditions, paralogyChoices, default="NA")
masterTableMouse["Paralogy Type"] = np.select(mouseParalogyConditions, paralogyChoices, default="NA")

# Create Master Tables

In [14]:
newColOrderHuman = list(masterTableHuman.columns[0:3]) + list(masterTableHuman.columns[-1:]) + list(masterTableHuman.columns[3:4]) + list(masterTableHuman.columns[-8:-3]) + list(masterTableHuman.columns[-3:-2]) + list(masterTableHuman.columns[-2:-3]) + list(masterTableHuman.columns[12:13]) + list(masterTableHuman.columns[21:22]) + list(masterTableHuman.columns[4:12]) + list(masterTableHuman.columns[13:21]) + list(masterTableHuman.columns[-2:-1])
newColOrderMouse = list(masterTableMouse.columns[0:3]) + list(masterTableMouse.columns[-1:]) + list(masterTableMouse.columns[3:4]) + list(masterTableMouse.columns[-8:-3]) + list(masterTableMouse.columns[-3:-2]) + list(masterTableMouse.columns[-2:-3]) + list(masterTableMouse.columns[12:13]) + list(masterTableMouse.columns[21:22]) + list(masterTableMouse.columns[4:12]) + list(masterTableMouse.columns[13:21]) + list(masterTableMouse.columns[-2:-1])

masterTableHuman = masterTableHuman.loc[:, newColOrderHuman]
masterTableMouse = masterTableMouse.loc[:, newColOrderMouse]

In [15]:
# masterTableHuman.to_csv("/Users/andrewhsu/Projects/McNair/data/humanParalogMasterTable.csv", index=False)
# masterTableHuman.to_parquet("/Users/andrewhsu/Projects/McNair/data/humanParalogMasterTable.parquet", index=False)

# masterTableMouse.to_csv("/Users/andrewhsu/Projects/McNair/data/mouseParalogMasterTable.csv", index=False)
# masterTableMouse.to_parquet("/Users/andrewhsu/Projects/McNair/data/mouseParalogMasterTable.parquet", index=False)

In [16]:
display(masterTableHuman)
display(masterTableMouse)

,Gene stable ID,Human paralogue gene stable ID_x,Human paralogue homology type,Paralogy Type,Paralogue last common ancestor with Human,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,...,Parental Stomach,Daughter Brain,Daughter Colon,Daughter Esophagus,Daughter Heart,Daughter Kidney,Daughter Liver,Daughter Pancreas,Daughter Stomach,Exon stable ID
0,ENSG00000210049,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,1.57815E+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSE00001544501
1,ENSG00000211459,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,4.34131E+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSE00001544499
2,ENSG00000210077,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,1.51203E+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSE00001544498
3,ENSG00000210082,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,1.98982E+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSE00001544497
4,ENSG00000209082,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,6.18579E+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSE00002006242
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3625954,ENSG00000157873,ENSG00000243509,other_paralog,out-paralog,Chordata,1.89904E+02,5.00351E-01,1.04461E+01,4.63805E-01,1.42857E-01,...,8.21904E+01,5.59910E+00,6.12330E+00,5.78953E+00,9.63287E-01,4.66856E+00,8.47602E-01,1.23503E+00,4.02125E+00,"ENSE00001759361, ENSE00001576845, ENSE00003529..."
3625955,ENSG00000157873,ENSG00000026103,other_paralog,out-paralog,Chordata,1.96173E+02,3.23165E-01,1.26015E+01,1.98451E-01,2.14286E-01,...,8.21904E+01,6.21104E-01,2.57837E+00,2.34953E+00,9.91792E-01,1.59063E+00,1.22803E+00,4.26707E-01,1.09465E+00,"ENSE00001759361, ENSE00001576845, ENSE00003529..."
3625956,ENSG00000157873,ENSG00000120949,other_paralog,out-paralog,Chordata,1.98703E+02,7.13202E-01,1.41740E+01,9.37936E-01,4.28571E-01,...,8.21904E+01,8.56435E-01,7.09645E-01,8.58565E-01,1.05779E+00,4.12732E-01,9.87493E-02,7.84178E-02,1.14326E+00,"ENSE00001759361, ENSE00001576845, ENSE00003529..."
3625957,ENSG00000132676,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,4.16859E+01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"ENSE00001936122, ENSE00001845528, ENSE00001811..."


,Gene stable ID,Mouse paralogue gene stable ID_x,Mouse paralogue homology type,Paralogy Type,Paralogue last common ancestor with Mouse,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,...,Parental Stomach,Daughter Brain,Daughter Colon,Daughter Esophagus,Daughter Heart,Daughter Kidney,Daughter Liver,Daughter Pancreas,Daughter Stomach,Exon stable ID
0,ENSMUSG00000064336,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,2.50860E+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSMUSE00000521514
1,ENSMUSG00000064337,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,1.46674E+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSMUSE00000521515
2,ENSMUSG00000064338,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,1.40266E+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSMUSE00000521516
3,ENSMUSG00000064339,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,1.33543E+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSMUSE00000521517
4,ENSMUSG00000064340,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,NaN,...,1.50967E+02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSMUSE00000521518
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2426744,ENSMUSG00000026833,ENSMUSG00000022026,other_paralog,out-paralog,Bilateria,9.16589E+02,1.31315E+00,1.00157E+01,1.09244E+00,2.50000E-01,...,6.50670E+00,2.13108E+00,1.56930E-01,1.67683E-01,1.70967E+00,1.56433E+01,2.19291E-01,4.11394E-01,6.71852E+00,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS..."
2426745,ENSMUSG00000026833,ENSMUSG00000027848,other_paralog,out-paralog,Bilateria,9.08057E+02,1.04051E+00,6.42890E+00,7.14240E-01,8.33333E-02,...,6.50670E+00,1.03947E+01,8.81571E+00,1.16197E+01,5.96584E+00,1.32532E+01,8.66962E-01,2.66533E-01,6.50374E+00,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS..."
2426746,ENSMUSG00000026833,ENSMUSG00000038463,other_paralog,out-paralog,Bilateria,9.12359E+02,1.27445E+00,7.38974E+00,1.09320E+00,8.33333E-02,...,6.50670E+00,6.20568E+00,1.79683E+01,3.18644E+01,6.97836E+00,1.88162E+00,2.66059E-01,6.78200E-01,7.19410E+00,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS..."
2426747,ENSMUSG00000026833,ENSMUSG00000046167,other_paralog,out-paralog,Bilateria,9.18024E+02,9.27514E-01,1.16915E+01,5.25570E-01,4.16667E-01,...,6.50670E+00,7.61520E-01,7.21297E-03,1.14298E+00,3.13731E-02,6.14808E-03,3.14910E-02,0.00000E+00,5.85445E-03,"ENSMUSE00001242819, ENSMUSE00001266606, ENSMUS..."


# Statistical Testing

In [17]:
statisticsDFHuman = pd.DataFrame({
    "Category": ["out-paralog", "in-paralog"],
    "n": 0.0,
    "Median": 0.0,
    "Mean": 0.0, 
    "Rho": 0.0,
    "Corr PValue" : 0.0
})

statisticsDFMouse = pd.DataFrame({
    "Category": ["out-paralog", "in-paralog"],
    "n": 0.0,
    "Median": 0.0,
    "Mean": 0.0, 
    "Rho": 0.0,
    "Corr PValue" : 0.0
})

In [18]:
sigHumanDF = pd.DataFrame({
    "Group 1": ["out-paralog"],
    "Group 2": ["in-paralog"],
    "PValue": 0.0
})

sigMouseDF = pd.DataFrame({
    "Group 1": ["out-paralog"],
    "Group 2": ["in-paralog"],
    "PValue": 0.0
})


In [19]:
dfListHuman = []
dfListMouse = []
for distMetric in masterTableHuman.columns[5:10]:
    nHumanArr = []
    nMouseArr = []
    medianHumanArr = []
    medianMouseArr = []
    meanHumanArr = []
    meanMouseArr = []
    corrHumanArr = []
    corrMouseArr = []
    pValueHumanArr = []
    pValueMouseArr = []

    for category in statisticsDFHuman["Category"]:
        filteredHumanDF = masterTableHuman[masterTableHuman["Paralogy Type"] == category][distMetric].dropna()
        filteredMouseDF = masterTableMouse[masterTableMouse["Paralogy Type"] == category][distMetric].dropna()

        nHumanArr.append(filteredHumanDF.shape[0])
        nMouseArr.append(filteredMouseDF.shape[0])
        medianHumanArr.append(filteredHumanDF.median())
        medianMouseArr.append(filteredMouseDF.median())
        meanHumanArr.append(filteredHumanDF.mean())
        meanMouseArr.append(filteredMouseDF.mean())

        corrHuman = stats.spearmanr(masterTableHuman[(masterTableHuman["Paralogy Type"] == category) & (~masterTableHuman[distMetric].isna())][distMetric], masterTableHuman[(masterTableHuman["Paralogy Type"] == category) & (~masterTableHuman[distMetric].isna())]["Number of Duplicates"])
        corrMouse = stats.spearmanr(masterTableMouse[(masterTableMouse["Paralogy Type"] == category) & (~masterTableMouse[distMetric].isna())][distMetric], masterTableMouse[(masterTableMouse["Paralogy Type"] == category) & (~masterTableMouse[distMetric].isna())]["Number of Duplicates"])
        corrHumanArr.append(corrHuman[0])
        corrMouseArr.append(corrMouse[0])
        pValueHumanArr.append(corrHuman[1])
        pValueMouseArr.append(corrMouse[1])

    statisticsDFHuman["n"] = nHumanArr
    statisticsDFHuman["Median"] = medianHumanArr
    statisticsDFHuman["Mean"] = meanHumanArr
    statisticsDFHuman["Rho"] = corrHumanArr
    statisticsDFHuman["Corr PValue"] = pValueHumanArr
    statisticsDFMouse["n"] = nMouseArr
    statisticsDFMouse["Median"] = medianMouseArr
    statisticsDFMouse["Mean"] = meanMouseArr
    statisticsDFMouse["Rho"] = corrMouseArr
    statisticsDFMouse["Corr PValue"] = pValueMouseArr

    dfListHuman.append(statisticsDFHuman.copy())
    dfListMouse.append(statisticsDFMouse.copy())

In [22]:
dfListHuman[0]

,Category,n,Median,Mean,Rho,Corr PValue
0,out-paralog,3328728,5.29812E-02,1.28672E+01,-2.72617E-01,0.00000E+00
1,in-paralog,8222,1.63692E-01,8.27490E+01,-8.79423E-02,1.36634E-15


In [20]:
dfListHumanSig = []
dfListMouseSig = []
for distMetric in masterTableHuman.columns[5:10]:
    pValueHumanArr = []
    pValueMouseArr = []
    for idx in range(len(sigHumanDF["Group 1"])):
        filteredHumanDF1 = masterTableHuman[masterTableHuman["Paralogy Type"] == sigHumanDF["Group 1"][idx]][distMetric].dropna()
        filteredHumanDF2 = masterTableHuman[masterTableHuman["Paralogy Type"] == sigHumanDF["Group 2"][idx]][distMetric].dropna()
        filteredMouseDF1 = masterTableMouse[masterTableMouse["Paralogy Type"] == sigMouseDF["Group 1"][idx]][distMetric].dropna()
        filteredMouseDF2 = masterTableMouse[masterTableMouse["Paralogy Type"] == sigMouseDF["Group 2"][idx]][distMetric].dropna()

        humanSig = stats.mannwhitneyu(filteredHumanDF1, filteredHumanDF2)[1]
        mouseSig = stats.mannwhitneyu(filteredMouseDF1, filteredMouseDF2)[1]

        pValueHumanArr.append(humanSig)
        pValueMouseArr.append(mouseSig)
    sigHumanDF["PValue"] = pValueHumanArr
    sigMouseDF["PValue"] = pValueMouseArr

    dfListHumanSig.append(sigHumanDF.copy())
    dfListMouseSig.append(sigMouseDF.copy())
        

In [21]:
with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/humanParalogStatistics.xlsx") as w:
    for idx, distMetric in enumerate(masterTableHuman.columns[5:10]):
        dfListHuman[idx].to_excel(w, sheet_name=distMetric, index=False)
        dfListHumanSig[idx].to_excel(w, sheet_name=distMetric, index=False, startrow=0, startcol=7)
    
with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/mouseParalogStatistics.xlsx") as w:
    for idx, distMetric in enumerate(masterTableMouse.columns[5:10]):
        dfListMouse[idx].to_excel(w, sheet_name=distMetric, index=False)
        dfListMouseSig[idx].to_excel(w, sheet_name=distMetric, index=False, startrow=0, startcol=7)